# DF Fluorosis Diagnosis — Complete Experiment Suite (v5)

**Purpose**: Re-run ALL experiments for the DF fluorosis paper, export per-sample predictions for figure generation, and upload results to Kaggle dataset.

**Setup**: DF task only, ViT-Base (google/vit-base-patch16-224-in21k), DFID dataset (200 images, 4-class).

**6 Phases**:
1. Main 5-mode comparison (CE, Cumulative, SORD, EDL, EDL+ORCU) — ~1.5h
2. Multi-Seed CE vs EDL (2 modes × 3 seeds) — ~2h
3. 5-Fold CV (CE, EDL, EDL+ORCU) — ~2h
4. Lambda Sweep EDL+ORCU (9 combos) — ~1.5h
5. Temperature Calibration on best EDL — ~10 min
6. Master summary + upload to Kaggle dataset `hgxiao/fluorosis-data-1-exp-v5`

**Total estimated runtime**: ~7h on T4 x2 GPU.

**Key feature**: Every model exports per-sample predictions (logits, probabilities, evidence, uncertainty, targets) for paper figures.

## 1. Environment Setup

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), \
    "Git clone failed! Check repo visibility."
print("Code cloned successfully.")

## 2. Master Setup

Import all dependencies, download pretrained weights, set paths. Run this once before any experiment.

In [ ]:
import os, sys, json, copy, time, itertools
from datetime import datetime
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

# Pre-download pretrained weights (internet available only during setup)
print("Downloading pretrained weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Weights cached.")

# Paths
CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis-data-1"
OUTPUT_DIR = "/kaggle/working"
EXPORT_DIR = "/kaggle/working/exports"
os.makedirs(EXPORT_DIR, exist_ok=True)
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders, DFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"
print(f"Device: {device} | GPU: {gpu_name}")
print(f"CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}")
print("Setup complete.")

## 3. Core Functions

Self-contained functions used by all experiment phases. Each function is defined once and reused.

In [ ]:
# ---- Evaluation: standard metrics ----
@torch.no_grad()
def evaluate(model, loader, temperature=1.0):
    """Compute all metrics over a dataloader."""
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None:
            all_alpha.append(alpha.cpu())
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha, dim=0) if all_alpha else None
    z = torch.cat(all_z, dim=0)
    targets = torch.cat(all_targets, dim=0)
    return compute_metrics(alpha, z, targets, temperature=temperature)


# ---- Per-sample prediction export ----
@torch.no_grad()
def export_predictions(model, loader, filepath):
    """Save per-sample predictions for figure generation.
    Exports: logits (N,K), probabilities (N,K), predicted (N,), targets (N,),
             evidence (N,K), alpha (N,K), uncertainty (N,).
    """
    model.eval()
    all_z, all_targets = [], []
    all_alpha = []  # Dirichlet concentration

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        all_z.append(z.cpu())
        all_targets.append(targets.cpu())
        if alpha is not None:
            all_alpha.append(alpha.cpu())

    z_all = torch.cat(all_z, dim=0)  # (N, K)
    targets_all = torch.cat(all_targets, dim=0)  # (N,)
    probs = F.softmax(z_all, dim=-1)
    preds = torch.argmax(probs, dim=-1)

    K = z_all.shape[-1]
    if all_alpha:
        alpha_all = torch.cat(all_alpha, dim=0)  # (N, K)
        S = alpha_all.sum(dim=-1)
        u = K / S  # (N,)
        evidence = alpha_all - 1.0  # (N, K)
    else:
        alpha_all = None
        evidence = torch.zeros_like(z_all)
        u = -torch.sum(probs * torch.log(probs + 1e-8), dim=-1) / torch.log(
            torch.tensor(K, dtype=torch.float32))

    export_dict = {
        "logits": z_all.numpy().astype(np.float16),
        "probabilities": probs.numpy().astype(np.float16),
        "predicted": preds.numpy().astype(np.int8),
        "targets": targets_all.numpy().astype(np.int8),
        "evidence": evidence.numpy().astype(np.float16),
        "alpha": alpha_all.numpy().astype(np.float16) if alpha_all is not None else None,
        "uncertainty": u.numpy().astype(np.float16),
    }
    np.savez_compressed(filepath, **{k: v for k, v in export_dict.items() if v is not None})
    return export_dict


# ---- Criterion getter ----
def get_criterion(mode, model=None, epochs=100):
    """Return loss function for given mode."""
    if mode == "ce":
        class CELoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = F.nll_loss(F.log_softmax(z, dim=-1), targets)
                return loss, {"stage": 0, "L_ce": loss.item()}
        return CELoss()

    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cum_fn = CumulativeLoss(num_classes=4)
        class CumLoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                ol = model.ordinal_logits if hasattr(model, 'ordinal_logits') else z
                loss = cum_fn(ol, targets)
                return loss, {"stage": 0, "L_cum": loss.item()}
        return CumLoss()

    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sord_fn = ORCULoss(num_classes=4, t=3.0, lambda_reg=0.0)
        class SORDLoss(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = sord_fn(z, targets)
                return loss, {"stage": 0, "L_sord": loss.item()}
        return SORDLoss()

    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        edl_fn = EDLLoss(num_classes=4, kl_lambda=0.1)
        class EDLLoss_(nn.Module):
            def forward(self, alpha, z, targets, epoch=None):
                loss = edl_fn(alpha, targets, epoch, epochs)
                return loss, {"stage": 0, "L_edl": loss.item()}
        return EDLLoss_()

    elif mode == "edl_orcu":
        return CombinedLoss(
            num_classes=4, lambda_orcu=0.5, lambda_kl=0.1, orcu_t=3.0,
            orcu_lambda_reg=0.01,
            stage_1_epochs=5, stage_2_epochs=30, total_epochs=epochs)

    else:
        raise ValueError(f"Unknown mode: {mode}")


# ---- Optimizer & scheduler builders ----
def build_opt_sched(model, epochs, lr_backbone=1e-4, lr_head=1e-3,
                    warmup_epochs=5, weight_decay=0.05):
    """Create optimizer and scheduler."""
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (bb if n.startswith("backbone") else hd).append(p)
    optimizer = optim.AdamW([
        {"params": bb, "lr": lr_backbone},
        {"params": hd, "lr": lr_head},
    ], weight_decay=weight_decay)
    ws = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cs = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[ws, cs], milestones=[warmup_epochs])
    return optimizer, scheduler


# ---- Core training function ----
def train_one(task, data_root, output_dir, mode="edl", epochs=100,
              batch_size=32, seed=42, eval_every=10, export_name=None):
    """Train one model. Returns flat test metrics dict.
    Also saves per-sample predictions to EXPORT_DIR. """
    torch.manual_seed(seed)
    np.random.seed(seed)
    os.makedirs(output_dir, exist_ok=True)

    # Data
    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)}")

    # Model
    model = build_model(task=task, mode=mode)
    model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params:,} params | Mode: {mode} | Type: {type(model).__name__}")

    # Criterion, optimizer, scheduler
    criterion = get_criterion(mode, model=model, epochs=epochs)
    optimizer, scheduler = build_opt_sched(model, epochs)

    # Training
    best_val_acc, best_state = 0.0, None
    for epoch in range(epochs):
        model.train()
        epoch_losses = {"L_ce": 0.0, "L_cum": 0.0, "L_sord": 0.0,
                        "L_edl": 0.0, "L_orcu": 0.0, "L_total": 0.0}
        epoch_stage, n_batches = 0, 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, loss_info = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_stage = loss_info.get("stage", 0)
            for k in epoch_losses:
                if k in loss_info:
                    epoch_losses[k] += loss_info[k]
            n_batches += 1

        scheduler.step()
        for k in epoch_losses:
            epoch_losses[k] /= max(n_batches, 1)

        if (epoch + 1) % eval_every == 0 or epoch == 0 or epoch == epochs - 1:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val_acc:
                best_val_acc = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())

            loss_key = next((k for k in ["L_total", "L_ce", "L_cum", "L_sord",
                                        "L_edl", "L_orcu"]
                            if epoch_losses.get(k, 0) != 0), "L_total")
            print(f"[E{epoch+1:3d}/{epochs}] S={epoch_stage} | "
                  f"{loss_key}={epoch_losses.get(loss_key, 0):.4f} | "
                  f"Acc={vm['acc']:.4f} F1={vm['macro_f1']:.4f} "
                  f"QWK={vm['qwk']:.4f} ECE={vm['ece']:.4f} "
                  f"Unim={vm['pct_unimodal']:.2%} U={vm.get('mean_uncertainty', 0):.3f}")

    # Load best and evaluate on test set
    model.load_state_dict(best_state)
    tm = evaluate(model, test_loader)

    # Save model checkpoint
    torch.save({"model_state_dict": best_state, "test_metrics": tm},
               os.path.join(output_dir, "best.pt"))

    # Export per-sample predictions
    ename = export_name or f"{task}_{mode}_seed{seed}"
    export_predictions(model, test_loader,
                       os.path.join(EXPORT_DIR, f"{ename}_preds.npz"))
    print(f"  Predictions exported: {ename}_preds.npz")

    # Flatten metrics for JSON
    flat = {}
    for k, v in tm.items():
        if isinstance(v, (float, int, bool)):
            flat[k] = v
        elif isinstance(v, np.ndarray):
            flat[k] = v.tolist()
    flat["best_val_acc"] = best_val_acc
    flat["n_params"] = n_params
    flat["seed"] = seed
    flat["mode"] = mode
    flat["export_name"] = ename

    with open(os.path.join(output_dir, "test_metrics.json"), "w") as f:
        json.dump(flat, f, indent=2)

    print(f"[Test] Acc={tm['acc']:.4f} F1={tm['macro_f1']:.4f} "
          f"QWK={tm['qwk']:.4f} ECE={tm['ece']:.4f} "
          f"Unim={tm['pct_unimodal']:.2%} U-ECE={tm.get('u_ece',0):.4f} "
          f"AUROC(u)={tm.get('auroc_u',0):.4f}")
    return flat

print("All core functions ready: evaluate, export_predictions, get_criterion, build_opt_sched, train_one")

## 4. Data Verification

Confirm DF dataset is accessible with correct class distribution.

In [ ]:
print("=== DF Data Verification ===")
for split in ["train", "val", "test"]:
    ds = DFDataset(DATA_ROOT, split=split)
    labels = np.array([ds[i][1] for i in range(len(ds))])
    dist = {f"class_{k}": int((labels == k).sum()) for k in range(4)}
    print(f"  {split:5s}: {len(ds):3d} samples | {dist}")
print("Data OK.")

---
## Phase 1: Main 5-Mode Comparison

DF task, ViT-Base backbone, 5 modes (CE, Cumulative, SORD, EDL, EDL+ORCU), seed=42, 100 epochs.

Expected: ~1.5h.

In [ ]:
MAIN_MODES = ["ce", "cumulative", "sord", "edl", "edl_orcu"]
main_results = {}
t0 = time.time()

for mode in MAIN_MODES:
    print(f"\n{'='*60}")
    print(f"Phase 1 — DF | Mode: {mode} | Seed: 42")
    print(f"{'='*60}")
    out_dir = os.path.join(OUTPUT_DIR, f"phase1_{mode}")
    metrics = train_one("df", DATA_ROOT, out_dir, mode=mode, epochs=100,
                        seed=42, export_name=f"phase1_{mode}")
    main_results[mode] = {k: v for k, v in metrics.items()
                          if isinstance(v, (float, int))}

with open(os.path.join(OUTPUT_DIR, "phase1_main_results.json"), "w") as f:
    json.dump(main_results, f, indent=2)

print(f"\nPhase 1 complete. Elapsed: {(time.time()-t0)/60:.1f} min")
print(f"\n{'='*80}")
print(f"DF Main Results (seed=42, 100 epochs)")
print(f"{'='*80}")
print(f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
print("-" * 78)
for mode, m in main_results.items():
    print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
          f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} "
          f"{m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
          f"{m.get('pct_unimodal',0):7.2%}")

---
## Phase 2: Multi-Seed Validation — CE vs EDL

2 modes (CE, EDL) × 3 seeds (42, 123, 456) = 6 runs, 100 epochs each.

Expected: ~2h.

In [ ]:
MULTISEED_MODES = ["ce", "edl"]
MULTISEED_SEEDS = [42, 123, 456]
multiseed_results = {}
t0 = time.time()

for mode in MULTISEED_MODES:
    for seed in MULTISEED_SEEDS:
        run_name = f"{mode}_seed{seed}"
        print(f"\n{'='*60}")
        print(f"Phase 2 — DF | Mode: {mode} | Seed: {seed}")
        print(f"{'='*60}")
        out_dir = os.path.join(OUTPUT_DIR, f"phase2_{run_name}")
        metrics = train_one("df", DATA_ROOT, out_dir, mode=mode, epochs=100,
                            seed=seed, export_name=f"phase2_{run_name}")
        multiseed_results[run_name] = {k: v for k, v in metrics.items()
                                       if isinstance(v, (float, int))}

with open(os.path.join(OUTPUT_DIR, "phase2_multiseed_results.json"), "w") as f:
    json.dump(multiseed_results, f, indent=2)

# Aggregate summary
SCALAR_METRICS = ["acc", "macro_f1", "qwk", "ece", "pct_unimodal", "u_ece", "auroc_u"]
print(f"\nPhase 2 complete. Elapsed: {(time.time()-t0)/60:.1f} min")
print(f"\n{'='*80}")
print(f"DF Multi-Seed Summary (mean ± std across 3 seeds)")
print(f"{'='*80}")
print(f"{'Metric':<18} {'CE':>20} {'EDL':>20}")
print("-" * 60)
for metric in SCALAR_METRICS:
    ce_vals = [multiseed_results[f"ce_seed{s}"][metric] for s in MULTISEED_SEEDS]
    edl_vals = [multiseed_results[f"edl_seed{s}"][metric] for s in MULTISEED_SEEDS]
    print(f"{metric:<18} {np.mean(ce_vals):7.4f}±{np.std(ce_vals):.4f}    "
          f"{np.mean(edl_vals):7.4f}±{np.std(edl_vals):.4f}")

# Per-seed breakdown
print(f"\n{'='*80}")
print("Per-Seed Breakdown")
print(f"{'='*80}")
for mode in MULTISEED_MODES:
    print(f"\n--- {mode.upper()} ---")
    print(f"{'Seed':<8} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'Unim':>8} {'U-ECE':>8} {'AUROC(u)':>8}")
    print("-" * 72)
    for seed in MULTISEED_SEEDS:
        r = multiseed_results[f"{mode}_seed{seed}"]
        print(f"{seed:<8} {r['acc']:8.4f} {r['macro_f1']:8.4f} "
              f"{r['qwk']:8.4f} {r['ece']:8.4f} "
              f"{r['pct_unimodal']:7.2%} {r['u_ece']:8.4f} "
              f"{r['auroc_u']:8.4f}")

# CV of key metrics
print(f"\n{'='*80}")
print("Cross-Seed CV (std/mean %, lower = more stable)")
print(f"{'='*80}")
for metric in ["acc", "macro_f1", "qwk"]:
    ce_cv = np.std([multiseed_results[f"ce_seed{s}"][metric] for s in MULTISEED_SEEDS]) / \
            np.mean([multiseed_results[f"ce_seed{s}"][metric] for s in MULTISEED_SEEDS]) * 100
    edl_cv = np.std([multiseed_results[f"edl_seed{s}"][metric] for s in MULTISEED_SEEDS]) / \
             np.mean([multiseed_results[f"edl_seed{s}"][metric] for s in MULTISEED_SEEDS]) * 100
    print(f"{metric:<18} CE={ce_cv:.1f}%  EDL={edl_cv:.1f}%")

---
## Phase 3: 5-Fold Cross-Validation

Stratified 5-fold CV for CE, EDL, EDL+ORCU on DF. 60 epochs per fold.

Expected: ~2h.

In [ ]:
CV_METHODS = ["ce", "edl", "edl_orcu"]
N_FOLDS = 5
CV_EPOCHS = 60
CV_SEED = 42

# Get labels for stratification
label_ds = DFDataset(DATA_ROOT, split="train", split_seed=CV_SEED)
labels = np.array([label_ds[i][1] for i in range(len(label_ds))])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=CV_SEED)
cv_fold_results = {m: [] for m in CV_METHODS}
t0 = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(label_ds)), labels)):
    print(f"\n{'#'*50}")
    print(f"# DF CV Fold {fold_idx+1}/{N_FOLDS}")
    print(f"{'#'*50}")

    # Build fold datasets
    train_ds = DFDataset(DATA_ROOT, split="train", split_seed=CV_SEED)
    val_ds = DFDataset(DATA_ROOT, split="train", split_seed=CV_SEED)
    test_ds = DFDataset(DATA_ROOT, split="test")
    train_ds.transform = get_transforms("df", is_train=True)
    val_ds.transform = get_transforms("df", is_train=False)
    test_ds.transform = get_transforms("df", is_train=False)

    train_subset = Subset(train_ds, train_idx)
    val_subset = Subset(val_ds, val_idx)

    train_loader = DataLoader(train_subset, batch_size=32, shuffle=True,
                              num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_subset, batch_size=32, shuffle=False,
                            num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=32, shuffle=False,
                             num_workers=2, pin_memory=True)

    for method in CV_METHODS:
        f_seed = CV_SEED + fold_idx * 100
        torch.manual_seed(f_seed)
        np.random.seed(f_seed)

        print(f"  [{method}] training fold {fold_idx+1}...")
        model = build_model(task="df", mode=method)
        model.to(device)

        criterion = get_criterion(method, model=model, epochs=CV_EPOCHS)
        optimizer, scheduler = build_opt_sched(model, CV_EPOCHS)

        best_val_acc, best_state = 0.0, None
        for epoch in range(CV_EPOCHS):
            model.train()
            for images, targets in train_loader:
                images, targets = images.to(device), targets.to(device)
                alpha, z = model(images)
                loss, _ = criterion(alpha, z, targets, epoch)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            scheduler.step()

            if (epoch + 1) % 10 == 0 or epoch == CV_EPOCHS - 1:
                vm = evaluate(model, val_loader)
                if vm["acc"] > best_val_acc:
                    best_val_acc = vm["acc"]
                    best_state = copy.deepcopy(model.state_dict())

        # Test on hold-out test set
        model.load_state_dict(best_state)
        tm = evaluate(model, test_loader)

        # Export per-sample predictions
        export_predictions(model, test_loader,
                           os.path.join(EXPORT_DIR, f"phase3_cv_{method}_fold{fold_idx}_preds.npz"))

        cv_fold_results[method].append({
            "fold": fold_idx,
            **{k: v for k, v in tm.items() if isinstance(v, (float, int))}
        })
        print(f"    Acc={tm['acc']:.4f} F1={tm['macro_f1']:.4f} "
              f"QWK={tm['qwk']:.4f} ECE={tm['ece']:.4f}")

# Aggregate CV results
cv_summary = {}
METRICS_FOR_CV = ["acc", "macro_f1", "qwk", "ece", "pct_unimodal", "u_ece", "auroc_u"]
for method in CV_METHODS:
    agg = {}
    for mn in METRICS_FOR_CV:
        vals = [f[mn] for f in cv_fold_results[method] if mn in f]
        agg[mn] = {"mean": float(np.mean(vals)), "std": float(np.std(vals)), "values": vals}
    cv_summary[method] = agg

# Paired t-tests
cv_significance = {}
for i, m1 in enumerate(CV_METHODS):
    for j, m2 in enumerate(CV_METHODS):
        if i >= j:
            continue
        pair_key = f"{m1}_vs_{m2}"
        cv_significance[pair_key] = {}
        for mn in ["acc", "macro_f1", "qwk", "ece"]:
            v1 = [f[mn] for f in cv_fold_results[m1]]
            v2 = [f[mn] for f in cv_fold_results[m2]]
            t_stat, p_val = stats.ttest_rel(v1, v2)
            cv_significance[pair_key][mn] = {
                "t_statistic": float(t_stat), "p_value": float(p_val),
                "significant": bool(p_val < 0.05)}

# Save
cv_output = {
    "methods": CV_METHODS, "n_folds": N_FOLDS, "epochs_per_fold": CV_EPOCHS,
    "fold_results": {m: cv_fold_results[m] for m in CV_METHODS},
    "summary": {m: {k: {"mean": v["mean"], "std": v["std"]}
                  for k, v in cv_summary[m].items()} for m in CV_METHODS},
    "significance": cv_significance,
}
with open(os.path.join(OUTPUT_DIR, "phase3_cv_results.json"), "w") as f:
    json.dump(cv_output, f, indent=2)

print(f"\nPhase 3 complete. Elapsed: {(time.time()-t0)/60:.1f} min")
print(f"\n{'='*80}")
print("DF 5-Fold CV Summary (mean ± std)")
print(f"{'='*80}")
for method in CV_METHODS:
    s = cv_summary[method]
    print(f"  {method:<14}: Acc={s['acc']['mean']:.4f}±{s['acc']['std']:.4f}  "
          f"QWK={s['qwk']['mean']:.4f}±{s['qwk']['std']:.4f}  "
          f"ECE={s['ece']['mean']:.4f}±{s['ece']['std']:.4f}")

print(f"\n  Significance (p < 0.05 marked *):")
for pair, tests in cv_significance.items():
    sigs = []
    for m, v in tests.items():
        if v["significant"]:
            sigs.append(f"{m}(p={v['p_value']:.3f})*")
        else:
            sigs.append(f"{m}(p={v['p_value']:.3f})")
    print(f"    {pair}: {', '.join(sigs)}")

---
## Phase 4: EDL+ORCU Lambda Sweep

Sweep λ_orcu ∈ {0.1, 0.3, 0.5} × λ_reg ∈ {0.005, 0.01, 0.02} = 9 combos, 50 epochs each.

Expected: ~1.5h.

In [ ]:
LAMBDA_ORCU_VALS = [0.1, 0.3, 0.5]
LAMBDA_REG_VALS = [0.005, 0.01, 0.02]
SWEEP_EPOCHS = 50
SWEEP_SEED = 42

train_loader, val_loader, test_loader = create_dataloaders(
    DATA_ROOT, task="df", batch_size=32, num_workers=2)

sweep_results = []
best_val_acc = 0.0
best_combo = None
t0 = time.time()

print(f"{'='*60}")
print(f"EDL+ORCU Lambda Sweep — DF")
print(f"  λ_orcu: {LAMBDA_ORCU_VALS}")
print(f"  λ_reg:  {LAMBDA_REG_VALS}")
print(f"  epochs: {SWEEP_EPOCHS} per run")
print(f"  total:  {len(LAMBDA_ORCU_VALS) * len(LAMBDA_REG_VALS)} runs")
print(f"{'='*60}")

for lam_orcu, lam_reg in itertools.product(LAMBDA_ORCU_VALS, LAMBDA_REG_VALS):
    print(f"\n  [λ_orcu={lam_orcu}, λ_reg={lam_reg}] training...")
    torch.manual_seed(SWEEP_SEED)

    model = build_model(task="df", mode="edl_orcu")
    model.to(device)

    criterion = CombinedLoss(
        num_classes=4, lambda_orcu=lam_orcu, lambda_kl=0.1, orcu_t=3.0,
        orcu_lambda_reg=lam_reg,
        stage_1_epochs=5, stage_2_epochs=15, total_epochs=SWEEP_EPOCHS)

    optimizer, scheduler = build_opt_sched(model, SWEEP_EPOCHS)

    best_val = 0.0
    best_state = None
    for epoch in range(SWEEP_EPOCHS):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()

        if (epoch + 1) % 10 == 0 or epoch == SWEEP_EPOCHS - 1:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val:
                best_val = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)
    tm = evaluate(model, test_loader)

    result = {
        "lambda_orcu": lam_orcu, "lambda_reg": lam_reg,
        "val_acc": best_val,
        "test_acc": tm["acc"], "test_qwk": tm["qwk"],
        "test_ece": tm["ece"], "test_unim": tm["pct_unimodal"],
        "test_u_ece": tm["u_ece"], "test_auroc_u": tm["auroc_u"],
        "test_f1": tm["macro_f1"],
    }
    sweep_results.append(result)

    if best_val > best_val_acc:
        best_val_acc = best_val
        best_combo = (lam_orcu, lam_reg)

    print(f"    Val Acc={best_val:.4f} | Test Acc={tm['acc']:.4f} "
          f"QWK={tm['qwk']:.4f} ECE={tm['ece']:.4f} "
          f"Unim={tm['pct_unimodal']:.2%}")

# Save best model from sweep
if best_combo:
    # Re-train with best combo (50 epochs, save predictions)
    print(f"\nRe-training best combo λ_orcu={best_combo[0]}, λ_reg={best_combo[1]} for export...")
    torch.manual_seed(SWEEP_SEED)
    model = build_model(task="df", mode="edl_orcu")
    model.to(device)
    criterion = CombinedLoss(
        num_classes=4, lambda_orcu=best_combo[0], lambda_kl=0.1, orcu_t=3.0,
        orcu_lambda_reg=best_combo[1],
        stage_1_epochs=5, stage_2_epochs=15, total_epochs=SWEEP_EPOCHS)
    optimizer, scheduler = build_opt_sched(model, SWEEP_EPOCHS)
    best_val, best_state = 0.0, None
    for epoch in range(SWEEP_EPOCHS):
        model.train()
        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha, z = model(images)
            loss, _ = criterion(alpha, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        scheduler.step()
        if (epoch + 1) % 10 == 0:
            vm = evaluate(model, val_loader)
            if vm["acc"] > best_val:
                best_val = vm["acc"]
                best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    export_predictions(model, test_loader,
                       os.path.join(EXPORT_DIR, "phase4_best_sweep_preds.npz"))
    torch.save({"model_state_dict": best_state},
               os.path.join(OUTPUT_DIR, "phase4_best_sweep.pt"))

with open(os.path.join(OUTPUT_DIR, "phase4_lambda_sweep.json"), "w") as f:
    json.dump({"results": sweep_results, "best_combo": list(best_combo) if best_combo else []}, f, indent=2)

print(f"\nPhase 4 complete. Elapsed: {(time.time()-t0)/60:.1f} min")
print(f"\n{'='*80}")
print("Lambda Sweep Results (sorted by Val Acc)")
print(f"{'='*80}")
print(f"{'λ_orcu':>8} {'λ_reg':>8} {'Val':>8} {'Test':>8} {'QWK':>8} {'F1':>8} {'ECE':>8} {'Unim':>8} {'U-ECE':>8} {'AUROC(u)':>8}")
print("-" * 88)
for r in sorted(sweep_results, key=lambda x: x["val_acc"], reverse=True):
    print(f"{r['lambda_orcu']:8.3f} {r['lambda_reg']:8.4f} "
          f"{r['val_acc']:8.4f} {r['test_acc']:8.4f} {r['test_qwk']:8.4f} {r['test_f1']:8.4f} "
          f"{r['test_ece']:8.4f} {r['test_unim']:7.2%} {r['test_u_ece']:8.4f} {r['test_auroc_u']:8.4f}")
print(f"\nBest: λ_orcu={best_combo[0]}, λ_reg={best_combo[1]} (Val Acc={best_val_acc:.4f})")

---
## Phase 5: Temperature Calibration

Load the best EDL model (from Phase 1 or Phase 2 seed=42) and sweep T ∈ {0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0}.

Expected: ~10 min.

In [ ]:
# Load best EDL model — try Phase 1 first, then Phase 2 seed 42
ckpt_path = None
for candidate in [
    os.path.join(OUTPUT_DIR, "phase1_edl", "best.pt"),
    os.path.join(OUTPUT_DIR, "phase2_edl_seed42", "best.pt"),
]:
    if os.path.exists(candidate):
        ckpt_path = candidate
        break

if ckpt_path is None:
    print("No EDL checkpoint found! Training EDL from scratch (100 epochs)...")
    out_dir = os.path.join(OUTPUT_DIR, "phase5_edl")
    train_one("df", DATA_ROOT, out_dir, mode="edl", epochs=100, seed=42,
              export_name="phase5_edl")
    ckpt_path = os.path.join(out_dir, "best.pt")

print(f"Loading EDL model from {ckpt_path}")
model = build_model(task="df", mode="edl")
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

_, _, test_loader = create_dataloaders(DATA_ROOT, task="df", batch_size=32, num_workers=2)

TEMPERATURES = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0]
temp_results = {}

print(f"\n{'='*60}")
print("Temperature Calibration — DF EDL")
print(f"{'='*60}")
print(f"{'T':>6} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
print("-" * 72)

for T in TEMPERATURES:
    m = evaluate(model, test_loader, temperature=T)
    temp_results[f"T_{T}"] = {
        "acc": m["acc"], "macro_f1": m["macro_f1"], "qwk": m["qwk"],
        "ece": m["ece"], "u_ece": m["u_ece"], "auroc_u": m["auroc_u"],
        "pct_unimodal": m["pct_unimodal"],
    }
    print(f"{T:6.1f} {m['acc']:8.4f} {m['macro_f1']:8.4f} {m['qwk']:8.4f} "
          f"{m['ece']:8.4f} {m['u_ece']:8.4f} {m['auroc_u']:8.4f} {m['pct_unimodal']:7.2%}")

best_ece = min(temp_results.items(), key=lambda x: x[1]["ece"])
best_auroc = max(temp_results.items(), key=lambda x: x[1]["auroc_u"])
print(f"\nBest ECE: {best_ece[0]} -> ECE={best_ece[1]['ece']:.4f}")
print(f"Best AUROC(u): {best_auroc[0]} -> AUROC(u)={best_auroc[1]['auroc_u']:.4f}")

with open(os.path.join(OUTPUT_DIR, "phase5_temperature_sweep.json"), "w") as f:
    json.dump(temp_results, f, indent=2)

print("\nPhase 5 complete.")

---
## Phase 6: Master Summary + Export to Kaggle Dataset

Compile all results, generate final tables, and upload to `hgxiao/fluorosis-data-1-exp-v5`.

In [ ]:
print("=" * 80)
print("MASTER SUMMARY — DF Fluorosis Diagnosis")
print(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)

master = {
    "timestamp": datetime.now().isoformat(),
    "task": "df",
    "dataset": "DFID (200 images, 4-class)",
    "backbone": "ViT-Base (google/vit-base-patch16-224-in21k)",
}

# Phase 1: Main results
p1_path = os.path.join(OUTPUT_DIR, "phase1_main_results.json")
if os.path.exists(p1_path):
    with open(p1_path) as f:
        master["phase1_main"] = json.load(f)
    print("\n--- Phase 1: Main 5-Mode Comparison (seed=42, 100ep) ---")
    print(f"{'Mode':<14} {'Acc':>8} {'F1':>8} {'QWK':>8} {'ECE':>8} {'U-ECE':>8} {'AUROC(u)':>8} {'%Unim':>8}")
    print("-" * 78)
    for mode, m in master["phase1_main"].items():
        print(f"{mode:<14} {m.get('acc',0):8.4f} {m.get('macro_f1',0):8.4f} "
              f"{m.get('qwk',0):8.4f} {m.get('ece',0):8.4f} "
              f"{m.get('u_ece',0):8.4f} {m.get('auroc_u',0):8.4f} "
              f"{m.get('pct_unimodal',0):7.2%}")

# Phase 2: Multi-seed
p2_path = os.path.join(OUTPUT_DIR, "phase2_multiseed_results.json")
if os.path.exists(p2_path):
    with open(p2_path) as f:
        master["phase2_multiseed"] = json.load(f)
    print("\n--- Phase 2: Multi-Seed CE vs EDL (3 seeds) ---")
    for mode in ["ce", "edl"]:
        vals = [master["phase2_multiseed"][f"{mode}_seed{s}"]["acc"] for s in [42, 123, 456]]
        print(f"  {mode}: {np.mean(vals):.4f} ± {np.std(vals):.4f}  (values: {[f'{v:.4f}' for v in vals]})")

# Phase 3: CV
p3_path = os.path.join(OUTPUT_DIR, "phase3_cv_results.json")
if os.path.exists(p3_path):
    with open(p3_path) as f:
        master["phase3_cv"] = json.load(f)
    print("\n--- Phase 3: 5-Fold CV ---")
    for method, s in master["phase3_cv"]["summary"].items():
        print(f"  {method:<14}: Acc={s['acc']['mean']:.4f}±{s['acc']['std']:.4f}  "
              f"QWK={s['qwk']['mean']:.4f}±{s['qwk']['std']:.4f}")

# Phase 4: Lambda sweep
p4_path = os.path.join(OUTPUT_DIR, "phase4_lambda_sweep.json")
if os.path.exists(p4_path):
    with open(p4_path) as f:
        master["phase4_lambda_sweep"] = json.load(f)
    ls = master["phase4_lambda_sweep"]
    print(f"\n--- Phase 4: Lambda Sweep (best: λ_orcu={ls['best_combo'][0]}, "
          f"λ_reg={ls['best_combo'][1]}) ---")
    for r in sorted(ls["results"], key=lambda x: x["val_acc"], reverse=True)[:5]:
        print(f"  λ_orcu={r['lambda_orcu']:.2f} λ_reg={r['lambda_reg']:.4f}: "
              f"Test Acc={r['test_acc']:.4f} QWK={r['test_qwk']:.4f}")

# Phase 5: Temperature
p5_path = os.path.join(OUTPUT_DIR, "phase5_temperature_sweep.json")
if os.path.exists(p5_path):
    with open(p5_path) as f:
        master["phase5_temperature"] = json.load(f)
    ts = master["phase5_temperature"]
    print("\n--- Phase 5: Temperature Calibration ---")
    for T_key in sorted(ts.keys(), key=lambda k: float(k.replace("T_", ""))):
        r = ts[T_key]
        print(f"  {T_key}: Acc={r['acc']:.4f} ECE={r['ece']:.4f} AUROC(u)={r['auroc_u']:.4f}")

# List exported prediction files
export_files = sorted([f for f in os.listdir(EXPORT_DIR) if f.endswith(".npz")])
print(f"\n--- Per-Sample Predictions Exported: {len(export_files)} files ---")
for f in export_files:
    fsize = os.path.getsize(os.path.join(EXPORT_DIR, f)) / 1024
    print(f"  {f} ({fsize:.1f} KB)")
master["export_files"] = export_files

# Save master summary
with open(os.path.join(OUTPUT_DIR, "phase6_master_summary.json"), "w") as f:
    json.dump(master, f, indent=2)

print(f"\nMaster summary saved to phase6_master_summary.json")
print("\n=== ALL EXPERIMENTS COMPLETE ===")

## 7. Upload Results to Kaggle Dataset `fluorosis-data-1-exp-v5`

This cell packages all results (metrics JSONs + per-sample prediction NPZs + model checkpoints) and creates/updates the Kaggle dataset.

In [ ]:
import os, json, shutil

PACKAGE_DIR = "/kaggle/working/fluorosis-data-1-exp-v5"
os.makedirs(PACKAGE_DIR, exist_ok=True)

# Collect all result files
collected = []

# Phase JSONs
for fname in ["phase1_main_results.json", "phase2_multiseed_results.json",
              "phase3_cv_results.json", "phase4_lambda_sweep.json",
              "phase5_temperature_sweep.json", "phase6_master_summary.json"]:
    src = os.path.join(OUTPUT_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(PACKAGE_DIR, fname))
        collected.append(fname)

# Best model checkpoints (Phase 1 for all 5 modes)
for mode in ["ce", "cumulative", "sord", "edl", "edl_orcu"]:
    src = os.path.join(OUTPUT_DIR, f"phase1_{mode}", "best.pt")
    if os.path.exists(src):
        dst = os.path.join(PACKAGE_DIR, f"checkpoint_{mode}.pt")
        shutil.copy2(src, dst)
        collected.append(f"checkpoint_{mode}.pt")

# Lambda sweep best checkpoint
src = os.path.join(OUTPUT_DIR, "phase4_best_sweep.pt")
if os.path.exists(src):
    shutil.copy2(src, os.path.join(PACKAGE_DIR, "checkpoint_best_sweep.pt"))
    collected.append("checkpoint_best_sweep.pt")

# Per-sample prediction NPZs
for fname in os.listdir(EXPORT_DIR):
    if fname.endswith(".npz"):
        src = os.path.join(EXPORT_DIR, fname)
        shutil.copy2(src, os.path.join(PACKAGE_DIR, fname))
        collected.append(fname)

# Create dataset metadata
metadata = {
    "title": "fluorosis-data-1-exp-v5",
    "id": "hgxiao/fluorosis-data-1-exp-v5",
    "licenses": [{"name": "CC0-1.0"}],
    "description": (
        "Complete DF fluorosis experiment results (v5). "
        "Contains: 5-mode comparison (CE/Cumulative/SORD/EDL/EDL+ORCU), "
        "multi-seed validation (CE vs EDL × 3 seeds), 5-fold CV, "
        "EDL+ORCU lambda sweep, temperature calibration, "
        "per-sample predictions for paper figures. "
        "All using ViT-Base (ImageNet-21k pretrained) on DFID dataset. "
        f"Generated {datetime.now().strftime('%Y-%m-%d')}."
    ),
}
with open(os.path.join(PACKAGE_DIR, "dataset-metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# Also copy the training notebook itself
notebook_src = "/kaggle/working/fluorosis_project/Kaggle_Setup/kaggle_train_v5.ipynb"
if os.path.exists(notebook_src):
    shutil.copy2(notebook_src, os.path.join(PACKAGE_DIR, "kaggle_train_v5.ipynb"))
    collected.append("kaggle_train_v5.ipynb")

print(f"Collected {len(collected)} files for dataset:")
for f in sorted(collected):
    fsize = os.path.getsize(os.path.join(PACKAGE_DIR, f)) / 1024
    if fsize > 1024:
        print(f"  {f} ({fsize/1024:.1f} MB)")
    else:
        print(f"  {f} ({fsize:.1f} KB)")

# Print the bash command for the user to run in a new cell or terminal
print(f"\n{'='*60}")
print("To create/update the Kaggle dataset, run in terminal:")
print(f"{'='*60}")
print(f"""
cd {PACKAGE_DIR}
kaggle datasets version -p . -m "v5: Complete DF experiment suite" --dir-mode zip
""")
print(f"\nOr for first-time creation:")
print(f"""
kaggle datasets create -p {PACKAGE_DIR} --dir-mode zip
""")

print(f"\nPackage ready at: {PACKAGE_DIR}")
print(f"Total package size: {sum(os.path.getsize(os.path.join(PACKAGE_DIR, f)) for f in collected) / 1024 / 1024:.1f} MB")

## 8. (Optional) Auto-Upload via Kaggle API

If `kaggle` CLI is available and authenticated, run this cell to auto-upload.

In [ ]:
import subprocess, os

PACKAGE_DIR = "/kaggle/working/fluorosis-data-1-exp-v5"

# Try to create or update the dataset
try:
    result = subprocess.run(
        ["kaggle", "datasets", "list", "--user", "hgxiao"],
        capture_output=True, text=True, timeout=30)
    if "fluorosis-data-1-exp-v5" in result.stdout:
        print("Dataset exists, creating new version...")
        cmd = ["kaggle", "datasets", "version", "-p", PACKAGE_DIR,
               "-m", f"v5: Complete DF experiment suite ({datetime.now().strftime('%Y-%m-%d')})"]
    else:
        print("Creating new dataset...")
        cmd = ["kaggle", "datasets", "create", "-p", PACKAGE_DIR, "--dir-mode", "zip"]
    
    subprocess.run(cmd, check=True, timeout=120)
    print("Dataset uploaded successfully!")
    print("URL: https://www.kaggle.com/datasets/hgxiao/fluorosis-data-1-exp-v5")
except subprocess.CalledProcessError as e:
    print(f"Upload failed: {e}")
    print("Please run the terminal commands printed in the previous cell manually.")
except FileNotFoundError:
    print("Kaggle CLI not available. Use the terminal commands from the previous cell.")
except Exception as e:
    print(f"Error: {e}")

---
## Experiment Checklist

| Phase | Experiment | Runtime | Status |
|-------|-----------|---------|--------|
| 1 | Main 5-mode (CE/Cumulative/SORD/EDL/EDL+ORCU) | ~1.5h | ⬜ |
| 2 | Multi-Seed CE vs EDL (2×3) | ~2h | ⬜ |
| 3 | 5-Fold CV (CE/EDL/EDL+ORCU) | ~2h | ⬜ |
| 4 | Lambda Sweep EDL+ORCU (9 combos) | ~1.5h | ⬜ |
| 5 | Temperature Calibration (7 temps) | ~10 min | ⬜ |
| 6 | Master Summary + Export to Kaggle | ~5 min | ⬜ |

**Run all cells in order. Each phase is self-contained — if a phase fails, fix and re-run just that cell.**